In [1]:
from datasets import load_dataset

ds = load_dataset("xu3kev/BIRD-SQL-data-train")

Generating train split:   0%|          | 0/9428 [00:00<?, ? examples/s]

In [4]:
ds

DatasetDict({
    train: Dataset({
        features: ['db_id', 'question', 'evidence', 'SQL', 'schema'],
        num_rows: 9428
    })
})

In [11]:
join_count = 0
total_count = 0
non_join_queries = []

for row in ds["train"]:
    sql = row["SQL"]
    total_count += 1
    if "JOIN" in sql.upper():
        join_count += 1
    else:
        non_join_queries.append(sql)

print(f"Number of SQL statements containing JOIN: {join_count}")
print(f"Total number of SQL statements: {total_count}")
print(f"Percentage: {(join_count/total_count)*100:.2f}%")

print("\nSample of queries without JOINs:")
for i, query in enumerate(non_join_queries[:3]):  # Show first 3 non-join queries
    print(f"\nQuery {i+1}:")
    print(query)

Number of SQL statements containing JOIN: 7212
Total number of SQL statements: 9428
Percentage: 76.50%

Sample of queries without JOINs:

Query 1:
SELECT COUNT(name) FROM director WHERE director = 'Wolfgang Reitherman'

Query 2:
SELECT COUNT(bioguide) FROM `current-terms` WHERE class IS NULL

Query 3:
SELECT character_id FROM paragraphs WHERE PlainText = 'O my poor brother! and so perchance may he be.'


In [13]:
import json
import re

def sql_to_weaviate_schema(sql_schema):
    # Parse CREATE TABLE statements
    table_pattern = r"CREATE TABLE (\w+)\s*\((.*?)\);"
    tables = re.findall(table_pattern, sql_schema, re.DOTALL)
    
    collections = []
    for table_name, table_content in tables:
        # Parse columns
        column_pattern = r"(\w+)\s+(INTEGER|TEXT|REAL)(?:\s+not null)?(?:\s+primary key)?(?:\s+default NULL)?"
        columns = re.findall(column_pattern, table_content)
        
        properties = []
        for col_name, col_type in columns:
            if col_name == 'id':  # Skip id columns as Weaviate handles these automatically
                continue
                
            data_type = ["number"] if col_type in ["INTEGER", "REAL"] else ["string"]
            
            properties.append({
                "name": col_name,
                "data_type": data_type,
                "description": f"The {col_name.replace('_', ' ')} of the {table_name}."
            })
            
        if properties:  # Only add tables with properties
            collections.append({
                "name": table_name.capitalize(),
                "properties": properties,
                "envisioned_use_case_overview": f"This collection stores information about {table_name.replace('_', ' ')}s."
            })
    
    return {
        "weaviate_collections": collections
    }

# Process first schema and save
for row in ds["train"]:
    schema = row["schema"]
    weaviate_schema = sql_to_weaviate_schema(schema)
    
    with open("bird-to-weaviate.json", "w") as f:
        json.dump(weaviate_schema, f, indent=2)
    break